#GCP Authentication & Configuration

In [ ]:
import os
from google.colab import auth, userdata

# --- Authentication ---
auth.authenticate_user()

# --- GCP Configuration ---
PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]

os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID


assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

add explanation

In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
TABLE = f"{PROJECT_ID}.tennis_raw.wta_matches"

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)
    return bq.query(sql, job_config=config).to_dataframe()

import pandas as pd
pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)

#Explore the raw data

## Preview the first few rows

In [ ]:
run(f"SELECT * FROM `{TABLE}` LIMIT 3").T

# Check features for missing values

Serve statistics are well filled in. Approx. 89% of matches have aces, double faults, serve points etc (only 10.9% empty).

Minutes is  41% empty - a fatigue feature based on minutes would only work for some matches. The score column i.  never empty, so games and sets could be drived from this and used to proxy fatigue.

Seed and entry columns missing values indicate empty unseeded or entered directly (raw data dictionary). To treat empty as "none" rather than as a gap. Height (10.5% and 14.9% empty) can be filled from the same player's other matches as it does not change.

Rank and ranking points are only 1.4% and 2.8% empty (probably unranked players)

In [ ]:
cols = [f.name for f in bq.get_table(TABLE).schema]
exprs = ",\n  ".join(
    f"ROUND(100 * COUNTIF(`{c}` IS NULL OR `{c}` = '') / COUNT(*), 1) AS `{c}`" for c in cols
)
missing = run(f"SELECT {exprs} FROM `{TABLE}`").T
missing.columns = ["percent_empty"]
missing.sort_values("percent_empty", ascending=False)

The number of rows BigQuery equal the sum of the individual file row counts. There are no duplicates (43,924 unique tournament and match-number pairs). Nothing  lost between the bucket and BigQuery

## Checking 'tourney_date' feature

Only one tourney_date per tournament. tourney-date gives the first day of each tournament, not the actual match date.

In [ ]:
run(f"""
SELECT
  COUNT(DISTINCT tourney_id) AS tournaments,
  COUNTIF(n_dates > 1) AS tournaments_with_more_than_one_date
FROM (
  SELECT tourney_id, COUNT(DISTINCT tourney_date) AS n_dates
  FROM `{TABLE}` GROUP BY tourney_id
)
""")

1,633 tournament distinct names far excceds the real number of tournaments. Level D (3,790 matches) is the team competition, Fed Cup (now Billie Jean King Cup), and each tie has its own name, like "Fed Cup WG R1: AUS vs ITA". Those ties are played in a team format at national venues - will likely exclude these.


In [ ]:
print(run(f"""
SELECT tourney_level, surface, COUNT(*) AS matches
FROM `{TABLE}` GROUP BY 1, 2 ORDER BY matches DESC
"""))

print(run(f"""
SELECT COUNT(DISTINCT tourney_name) AS distinct_names,
       COUNT(DISTINCT tourney_id) AS distinct_tourney_ids
FROM `{TABLE}`
"""))

add explanation

In [ ]:
run(f"""
SELECT COUNT(*) AS rows_total,
       COUNT(DISTINCT CONCAT(tourney_id, '|', match_num)) AS unique_matches
FROM `{TABLE}`
""")


Level codes to check:
O (probably Olympics)
50+H and 35+H (probably senior events)
W (?)


Cleaning to do later:
surface has both Clay and clay
115 Fed Cup rows have no surface.

Decisions to make:
Keep levels G, PM, P, I and F (approx. 39,300 matches)
Potentially to exclude Fed Cup, Olympics the senior events
To include or exlude W? Decide once sure what it represents


#IMPORTANT OBSERVATIONS RE POTENTIAL DATA LEAKAGE TO AVOID

NB: data has a "winner" and a "loser" side for each match. If  build features directly from that, the label is built into the layout, which is a leakage trap. To reshape it into player A and player B, assigned in a way unrelated to who won.

NB: the w_ and l_ statistics, minutes and score, are outcomes of the match itself, so they can never be features for that same match. They can only be used through earlier matches (rolling averages).

*******************************************************************

Examining how well features are filled in by year:

In [ ]:
print(run(f"""
SELECT tourney_level, COUNT(*) AS matches,
       ROUND(100*COUNTIF(w_svpt IS NULL)/COUNT(*),1) AS pct_no_serve_stats,
       ROUND(100*COUNTIF(minutes IS NULL)/COUNT(*),1) AS pct_no_minutes
FROM `{TABLE}` GROUP BY tourney_level ORDER BY matches DESC
"""))

print(run(f"""
SELECT SUBSTR(tourney_date,1,4) AS year, COUNT(*) AS matches,
       ROUND(100*COUNTIF(w_svpt IS NULL)/COUNT(*),1) AS pct_no_serve_stats,
       ROUND(100*COUNTIF(minutes IS NULL)/COUNT(*),1) AS pct_no_minutes
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I','F')
GROUP BY year ORDER BY year
"""))

In [ ]:
print(run(f"""
SELECT tourney_level,
       COUNT(DISTINCT tourney_id) AS tourney_ids,
       COUNT(DISTINCT SUBSTR(tourney_id, 6)) AS distinct_id_suffixes,
       COUNT(DISTINCT tourney_name) AS distinct_names
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I','F')
GROUP BY tourney_level ORDER BY tourney_ids DESC
"""))

run(f"""
SELECT SUBSTR(tourney_id, 6) AS id_suffix,
       COUNT(DISTINCT SUBSTR(tourney_id, 1, 4)) AS years,
       ARRAY_AGG(DISTINCT tourney_name LIMIT 4) AS example_names
FROM `{TABLE}` WHERE tourney_level IN ('G','PM','P','I','F')
GROUP BY id_suffix ORDER BY years DESC LIMIT 15
""")



Minutes is missing for essentially every match from 2010 to 2015 and is filled in from 2016 onward (0 to 7% empty). So minutes is a 2016+ feature only.

Serve stats are 10 to 13% empty in 2010 to 2014, worse in 2015 (22%), then almost complete from 2016 (0 to 2%).

2026 dips again (12.6% empty for serve stats), probably because the latest matches haven't been filled in yet. If test set includes 2026, the missingness there will differ from the earlier years.

This gap follows time, not chance. If we fill the gaps and train across the break, the model could pick up the year from which columns exist, which is a form of leakage.

Fed Cup (level D) is confirmed as a poor fit. 62% of its matches have no serve stats, and it's a team format at neutral venues. To exclude it.

Modelling levels G, PM, P, I and F give 39,342 matches (about 24,600 from 2016 onward). Both are above your 10,000 minimum. The small levels (W 321, O 256, 50+H 184, 35+H 31) are worth identifying but probably excluded.

Tournament IDs: The ID suffix (e.g. 580 for the Australian Open) is stable year to year, but it isn't a reliable location key:

Some suffixes cover different cities. Suffix 1056 maps to both Tokyo and Osaka, since the same licence moved.
Other tournaments change suffix over time. At level I there are 278 suffixes but only 109 names, so names are more stable than IDs.
Names are also spelt inconsistently ("US Open" and "Us Open").

This means the venue table should be keyed on the cleaned tournament name (lowercased and trimmed), not the ID. Aprox. 190 names across the modelling levels (before merging spelling variants).

Data has no indoor/outdoor/roof column, which we'd want for weather. Potentially add to the venue table by hand.



# Insights from origin data dictionary

# to go over/revise this
Level codes: G, PM, P, I and F are documented (Slams, Premier Mandatory, Premier, International, season-ending finals), and D is the team competition. This supports the modelling scope of G, PM, P, I and F.

Draw size: it's often rounded up to the next power of 2 (a 28-player draw appears as 32). We'll need that for estimating match days from rounds.

Ranks and ages: rank is the most recent ranking on or before the tournament's start date, and age is also as of the start date. Both are known before the match, so they're safe as features.

Seed and entry: these are known before the match. An empty seed means "not seeded", and entry codes include WC (wild card), Q (qualifier), LL (lucky loser) and PR (protected ranking).

New warnings from it
match_num can't be used to order matches. It sometimes counts down from 300, and sometimes it's arbitrary. Don't treat it as a time order. For ordering matches we need the estimated date plus round.

Anything about the match itself is off-limits as a feature for that match: score, minutes, and all the w_ and l_ statistics. They only enter the model as history, such as a player's rolling averages from earlier matches.

The stats are labelled "winner" and "loser". If we build features straight from those columns, the label leaks into the layout. In staging we'll reshape each match into player A and player B, assigned by a rule unrelated to who won.
What it doesn't cover

The dictionary doesn't define W, O, 50+H or 35+H. Two educated guesses, neither confirmed:

50+H and 35+H follow the ITF pattern (prize money in thousands), and I believe the "+H" means the event provides hospitality for players. They're small (184 and 31 matches) and are lower-tier ITF events.
W may be WTA 125 events (321 matches), and O may be the Olympics (256 matches). I don't know either, and I'd rather see the tournament names than guess.



Keep the dictionary with project
In repo, add a docs/ note that links to the dictionary file in the mirror at pinned commit, rather than copying the whole text. If  do copy it, keep the attribution to Jeff Sackmann and the CC BY-NC-SA licence with it.



In [ ]:
run(f"""
SELECT tourney_level, tourney_name, COUNT(*) AS matches,
       MIN(tourney_date) AS first_date, MAX(tourney_date) AS last_date
FROM `{TABLE}`
WHERE tourney_level IN ('W','O','50+H','35+H')
GROUP BY 1, 2 ORDER BY tourney_level, matches DESC
""")

In [ ]:
venues = run(f"""
SELECT LOWER(TRIM(tourney_name)) AS name_key,
       ARRAY_AGG(DISTINCT tourney_name LIMIT 5) AS spellings,
       ARRAY_AGG(DISTINCT tourney_level) AS levels,
       ARRAY_AGG(DISTINCT LOWER(surface) IGNORE NULLS) AS surfaces,
       COUNT(DISTINCT tourney_id) AS editions,
       MIN(SUBSTR(tourney_date, 1, 4)) AS first_year,
       MAX(SUBSTR(tourney_date, 1, 4)) AS last_year
FROM `{TABLE}`
WHERE tourney_level IN ('G','PM','P','I','F')
GROUP BY name_key
ORDER BY editions DESC
""")
print(len(venues), "distinct names")
venues


Small levels:
35+H and 50+H: the names start with W35+H and W50+H, so they're lower-tier ITF women's events. These are small, mostly single-edition events from 2025 (306 matches in total).
O: the Olympics, as I guessed (2012 and 2016 together, then Tokyo 2021 and Paris 2024).
W: these look like WTA 125 events (Parma, Bad Homburg, Strasbourg, Buenos Aires 125 and so on), the second tier below the main tour. Mostly one edition each, clustered in 2021.
Oddity: one W row is "WTA Finals" from 2015 (15 matches), which looks like a mislabelled season-ending event that should be under F.

Decision:
Model on levels G, PM, P, I and F and leave out D, O, W, 35+H and 50+H.

Reasons:
The excluded levels total about 4,580 matches, and most are one-off editions with too little history to learn from.
The Olympics and team events have different formats, venues and player priorities.
The remaining 39,342 matches are well above your 10,000 minimum.

Note to write this down as a decision with the reasons

In [ ]:
run(f"""
SELECT tourney_level, tourney_name, tourney_date, COUNT(*) AS matches
FROM `{TABLE}`
WHERE tourney_level = 'F'
   OR LOWER(tourney_name) LIKE '%finals%'
   OR LOWER(tourney_name) LIKE '%championships%'
GROUP BY 1, 2, 3
ORDER BY tourney_date
""")